# MedScan-AI Trainer

**Flagship Project:** Multi-modality medical image classification

**Data:** Chest X-ray (Normal vs Abnormal)

**Models:** Custom CNN + EfficientNet-B0 comparison

**Explainers:** Grad-CAM, Grad-CAM++, Score-CAM, Saliency, Guided Backprop, Integrated Gradients, Occlusion

In [ ]:
# Setup
import sys
sys.path.insert(0, '../..')

import torch
import numpy as np
import matplotlib.pyplot as plt

from shared.config import DEVICE, BATCH_SIZE, EPOCHS, LEARNING_RATE, EARLY_STOPPING_PATIENCE
from shared.models import create_model
from shared.pipelines.dataset import create_dataloaders
from shared.pipelines.train import train_model
from shared.utils.metrics import compute_metrics

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
# Load configuration
from config import PROJECT_ID, MODEL_NAME, CLASSES, IMG_SIZE
from shared.config import get_trained_model_path

checkpoint_path = str(get_trained_model_path(PROJECT_ID))
print(f'Project: {PROJECT_ID}')
print(f'Classes: {CLASSES}')
print(f'Model: {MODEL_NAME}')
print(f'Checkpoint will be saved to: {checkpoint_path}')

In [ ]:
# Load dataset
train_loader, val_loader, test_loader, ds_classes = create_dataloaders(
    data_root='data',
    class_names=CLASSES,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

print(f'Train: {len(train_loader.dataset)} images')
print(f'Val:   {len(val_loader.dataset)} images')
print(f'Test:  {len(test_loader.dataset)} images')
print(f'Classes: {ds_classes}')

## Model 1: Custom CNN

In [ ]:
# Create Custom CNN
custom_cnn = create_model(
    'custom_cnn',
    num_classes=len(CLASSES),
    in_channels=3,
    base_filters=32,
    num_blocks=4,
    dropout=0.5,
)

params = sum(p.numel() for p in custom_cnn.parameters())
print(f'Custom CNN: {params:,} total parameters')
print(custom_cnn)

In [ ]:
# Train Custom CNN
history_cnn = train_model(
    model=custom_cnn,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    num_epochs=EPOCHS,
    lr=LEARNING_RATE,
    patience=EARLY_STOPPING_PATIENCE,
    checkpoint_path=checkpoint_path,
)
print('Custom CNN training complete!')

In [ ]:
# Evaluate Custom CNN
custom_cnn.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True))
custom_cnn.eval()

all_p, all_l, all_pr = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = custom_cnn(images)
        probs = torch.softmax(outputs, dim=1)
        preds = probs.argmax(dim=1)
        all_p.extend(preds.cpu().numpy())
        all_l.extend(labels.cpu().numpy())
        all_pr.extend(probs.cpu().numpy())

m = compute_metrics(all_l, all_p, all_pr)
print('Test Results - Custom CNN:')
print(f'  Accuracy:  {m["accuracy"]:.4f}')
print(f'  Precision: {m["precision"]:.4f}')
print(f'  Recall:    {m["recall"]:.4f}')
print(f'  F1-Score:  {m["f1_score"]:.4f}')
if m.get('roc_auc'):
    print(f'  ROC-AUC:   {m["roc_auc"]:.4f}')

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, key, title in [(axes[0], 'accuracy', 'Accuracy'), (axes[1], 'loss', 'Loss')]:
    train_vals = [m[key] for m in history_cnn['train']]
    val_vals = [m[key] for m in history_cnn['val']]
    ax.plot(train_vals, label='Train', linewidth=2)
    ax.plot(val_vals, label='Val', linewidth=2)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_val = max([m['accuracy'] for m in history_cnn['val']])
print(f'Best Validation Accuracy: {best_val:.4f}')

## Model 2: EfficientNet-B0 (Comparison)

In [ ]:
# Create and train EfficientNet
checkpoint_eff = checkpoint_path.replace('.pth', '_efficientnet.pth')

efficientnet = create_model('efficientnet_b0', num_classes=len(CLASSES))
params_eff = sum(p.numel() for p in efficientnet.parameters())
print(f'EfficientNet-B0: {params_eff:,} parameters')

history_eff = train_model(
    model=efficientnet,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    num_epochs=EPOCHS,
    lr=1e-4,
    patience=EARLY_STOPPING_PATIENCE,
    checkpoint_path=checkpoint_eff,
)
print('EfficientNet training complete!')

In [ ]:
# Compare both models
cnn_best_val = max([m['accuracy'] for m in history_cnn['val']])
eff_best_val = max([m['accuracy'] for m in history_eff['val']])

print('=' * 60)
print('  Model Comparison')
print('=' * 60)
print(f'{"Model":<20} {"Params":<10} {"Test Acc":<10} {"Best Val":<10}')
print('-' * 50)
print(f'{"Custom CNN":<20} {params:<10,} {m["accuracy"]:<10.4f} {cnn_best_val:<10.4f}')

# Evaluate EfficientNet on test
efficientnet.load_state_dict(torch.load(checkpoint_eff, map_location=DEVICE, weights_only=True))
efficientnet.eval()
all_p, all_l = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = efficientnet(images)
        preds = outputs.argmax(dim=1)
        all_p.extend(preds.cpu().numpy())
        all_l.extend(labels.cpu().numpy())

from sklearn.metrics import accuracy_score
acc_eff = accuracy_score(all_l, all_p)
print(f'{"EfficientNet-B0":<20} {params_eff:<10,} {acc_eff:<10.4f} {eff_best_val:<10.4f}')

## XAI Explainability Demo

In [ ]:
# Run all 7 XAI explainers on a test image
from shared.explainers.xai_factory import run_all_explainers
from shared.pipelines.transforms import get_inference_transform
from PIL import Image
import os, random

test_dir = 'data/test'
if os.path.isdir(test_dir):
    cls_dir = random.choice([d for d in os.listdir(test_dir) if os.path.isdir(os.path.join(test_dir, d))])
    img_name = random.choice(os.listdir(os.path.join(test_dir, cls_dir)))
    img_path = os.path.join(test_dir, cls_dir, img_name)
    print(f'Sample: {img_path}')

    pil = Image.open(img_path).convert('RGB')
    transform = get_inference_transform(IMG_SIZE)
    tensor = transform(pil).unsqueeze(0).to(DEVICE)

    result = run_all_explainers(
        custom_cnn, tensor, CLASSES, device=DEVICE,
        occlude_size=32, occlude_stride=16,
    )

    pred = result['predictions']
    print(f'\nPrediction: {pred["predicted_class"]}')
    print(f'Confidence: {pred["confidence"]:.2%}')
    print(f'Explainers: {len(result["explanations"])}')

In [ ]:
# Display XAI gallery
from IPython.display import display, HTML

html = '<h2>XAI Heatmap Gallery</h2>'
html += '<div style="display:grid; grid-template-columns:repeat(3,1fr); gap:15px;">'
for key, exp in result['explanations'].items():
    if 'overlay_base64' in exp:
        html += '<div style="border:1px solid #ddd; border-radius:8px; padding:10px; text-align:center;">'
        html += f'<h4 style="margin:0;">{exp["label"]}</h4>'
        html += f'<img src="data:image/png;base64,{exp["overlay_base64"]}" style="width:100%;"/>'
        html += '</div>'
html += '</div>'
display(HTML(html))

In [ ]:
# Confusion Matrix
from shared.utils.visualization import plot_confusion_matrix
from sklearn.metrics import confusion_matrix
import base64
from IPython.display import Image as IPImage

custom_cnn.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE, weights_only=True))
custom_cnn.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = custom_cnn(images)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
cm_b64 = plot_confusion_matrix(cm, CLASSES)
display(IPImage(base64.b64decode(cm_b64)))

---
**Summary:**

Both models trained and evaluated. Weights saved to trained_models/.

Run 'python run.py' from project root to start the web interface
and upload a chest X-ray to see predictions with 7 XAI heatmaps.